# Deconvolution

In [1]:
from IPython.display import display
import pandas as pd

from dualde import load_example
from dualde.deconvolution import deconvolve_evo_vs_stability, merge_evo_spurs_dms
from dualde.scores import ESM2_650M_COLUMN, add_neg_spurs, load_spurs_table

EXAMPLE, cfg = load_example()
OUT = EXAMPLE / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
PDB_ID = cfg["pdb_id"]

evo = pd.read_csv(EXAMPLE / "scores" / "evolutionary.csv")
spurs = add_neg_spurs(load_spurs_table(EXAMPLE / "scores" / f"{PDB_ID}_spurs.csv"))
activity = pd.read_csv(EXAMPLE / "dms" / "activity.csv")
stability = pd.read_csv(EXAMPLE / "dms" / "stability.csv")
evo.to_csv(OUT / "evolutionary.csv", index=False)
spurs.to_csv(OUT / f"{PDB_ID}_spurs.csv", index=False)

merged = merge_evo_spurs_dms(evo, spurs, activity, stability)
evo_cols = [c for c in (ESM2_650M_COLUMN, "EVmutation") if c in merged.columns]
merged = deconvolve_evo_vs_stability(merged, evo_cols=evo_cols, stability_cols=("neg_spurs", "stability"))
merged.to_csv(OUT / "deconvolution.csv", index=False)

score_col = cfg["score_col"]
print(merged.shape, score_col)
display(merged[score_col].describe())


(2844, 29) ESM2_650M_neg_spurs_model


count    2724.000000
mean        0.491033
std         0.290456
min         0.000379
25%         0.243009
50%         0.494511
75%         0.740720
max         1.000000
Name: ESM2_650M_neg_spurs_model, dtype: float64